# NVIDIA LocateAnything - CONDA [Images]

El siguiente notebook es una adaptación junto a una breve documentación de los elementos compartidos por el equipo de LocateAnything mediante su GitHub [https://github.com/NVlabs/Eagle/tree/main/Embodied] para hacer uso del modelo de forma nativa, ellos comparten la forma de descargar e implementar las inferencias del modelo así como la recuperación de las inferencias para crear las bounding boxes, mi breve aportación recae en la recuperación de estos elementos para su representación visual mediante Matplotlib. 

En este primer acercamiento se usarán imágenes como uno de los dos inputs que solicita el modelo, pero el objetivo final de esta serie de aproximaciones es implementarlo en video.

## Instalar el modelo LocateAnything

In [ ]:
# Instalar el modelo LocateAnything-3B y sus dependencias a través de GitHub para su ejecución local
!git clone https://github.com/NVlabs/Eagle.git eagle
%cd eagle/Embodied
!pip install -e .

## NOTA IMPORTANTE: Debido al uso de librerías como triton, el uso de este modelo se ve limitado a los
#                   sistemas de Linux (en el cual se desarrolló) y Windows, Mac no cuenta con soporte o 
#                   una librería sustituta para este modelo, por lo que no funcionará nativamente (la única
#                   forma, pero limitada es haciendo uso de Google Colab).

## Librerías adicionales para ejecutar las inferencias

In [ ]:
# Importar librerías adicionales para hacer uso de LocateAnything
import re
import torch
from locateanything_worker import LocateAnythingWorker
from PIL import Image, ImageDraw             # ImageDraw es adicional, empleada para representar visualmente los resultados

# Importar librerías adicionales para su representación visual
import matplotlib.pyplot as plt

## Ejecución del modelo LocateAnything y métodos 

In [ ]:
worker = LocateAnythingWorker("nvidia/LocateAnything-3B")         # Importar el modelo LocateAnything-3B
img = Image.open('example.jpg').convert("RGB")                    # Importar la imagen a analizar

#### Object detection
Devuelve los bounding boxes de los objetos encontrados, útil para manejar detección de clases con exactitud.

In [ ]:
# answer_od = worker.detect(img, ["truck"])["answer"]
# print(answer_od)

#### Phrase Grounding
No empleas clases, sino una descripción de lo que se busca, permitiendo encontrar objetos para los que no se entrenó, pero para los que se les asigna una descripción como entrada.

In [ ]:
answer_pg = worker.ground_multi(img, "bushes on the street")["answer"]
print(answer_pg)

#### Scene Text Detection
Encuentra texto, líneas de texto en la imagen.

In [ ]:
# answer_std = worker.detect_text(img)["answer"]
# print(answer_std)

#### GUI Grounding (point)
Identifica elementos en interfaces gráficas.

In [ ]:
# answer_guigp = worker.ground_gui(img, "the search button", output_type="point")["answer"]
# print(answer_guigp)

#### Pointing
En lugar de una caja, devuelve un punto del objeto buscado por descripción. 

In [ ]:
# answer_p = worker.point(img, "the red car")["answer"]
# print(answer_p)

## Análisis de resultados del modelo y su interpretación en pixeles para su uso

#### RESULTADOS: Bounding Boxes

La función _parse_boxes_ de LocateAnything nos permite interpretar el resultado de la prompt solicitada, cuya estructura es una línea de texto que contiene la detección y las coordenadas de la misma mediante la normalización de estos valores del resultado y la interpretación de los mismos para su uso en el trazo de las Bounding Boxes que encierran al objeto solicitado. 
Esta función se emplea para interpretar los métodos: _Detect_, _Phrase Grounding_ y _Scene Text Detection_.


In [ ]:
w, h = img.size                                              # Extracción del ancho y alto de la imagen
boxes = LocateAnythingWorker.parse_boxes(answer_pg, w, h)    # DETECCIONES: Phrase Grounding

## ======= Otras interpretaciones para los otros métodos ======= ##

#boxes = LocateAnythingWorker.parse_boxes(answer_od, w, h)   # DETECCIONES: Object Detection
#boxes = LocateAnythingWorker.parse_boxes(answer_std, w, h)  # DETECCIONES: Scene Text Detection

print("Parsed boxes:", boxes)                                # Imprime los resultados 

#### RESULTADOS: Puntos

La función _parse_points_ de LocateAnything nos permite interpretar el resultado de la prompt solicitada, cuya estructura es una línea de texto que contiene la detección y las coordenadas de la misma mediante la normalización de estos valores del resultado y la interpretación de los mismos para su uso al generar un punto que señala el objeto solicitado.
Esta función se emplea para interpretar los métodos: _GUI Grounding_ y _Pointing_.

In [ ]:
w, h = img.size                                                 # Extracción del ancho y alto de la imagen
points = LocateAnythingWorker.parse_points(answer_guigp, w, h)  # DETECCIONES: Pointing

## ======= Otras interpretaciones para los otros métodos ======= ##

#points = LocateAnythingWorker.parse_boxes(answer_od, w, h)     # DETECCIONES: GUI Grounding

print("Parsed points:", points)                                 # Imprime los resultados 

## Visualización de resultados

#### Visualización para Bounding Boxes

A partir de la inferencia textual de las bounding boxes interpretadas por la función _parse_boxes_ generamos una interpretación visual de las cajas sobre la imagen para visualizar las inferencias del modelo, mediante la ayuda de _matplotlib_ e _ImageDraw_.

In [ ]:
img_draw = img.copy()                           # Crea una copia de la imagen original para visualizar resultados
draw = ImageDraw.Draw(img_draw)                 # Crea un objeto para editar / dibujar sobre la imagen

for box in boxes:                               # Recorre todas las inferencias encontradas por el modelo
    
                                                # NORMALIZACIÓN DE COORDENADAS DE LA CAJA
    # Coordenadas en X
    x1 = min(box["x1"], box["x2"])
    x2 = max(box["x1"], box["x2"])

    # Coordenadas en Y
    y1 = min(box["y1"], box["y2"])
    y2 = max(box["y1"], box["y2"])
                                                # Dibuja la caja / rectángulo a partir de los puntos normalizados
    draw.rectangle(
        [(x1, y1), (x2, y2)],
        outline="red",                          # Color de la caja (puede modificarse a conveniencia)
        width=2                                 # Grosor del trazo (puede modificarse a conveniencia)
    )

plt.figure(figsize=(12,12))                     # Mostrar la imagen con los resultados
plt.imshow(img_draw)
plt.axis("off")
plt.show()

#### Visualización para Puntos

A partir de la inferencia textual de los points interpretados por la función _parse_points_ generamos una interpretación visual de los puntos sobre la imagen para visualizar las inferencias del modelo, mediante la ayuda de _matplotlib_ e _ImageDraw_. Debido a la escala del mismo, se hizo uso de la figura elipse disponible en _ImageDraw_ para crear un elemento visual más identificable (en cuanto a dimensión), empleando las mismas coordenadas del punto o los puntos dados por la inferencia.

In [ ]:
img_draw = img.copy()                         # Crea una copia de la imagen original para visualizar resultados
draw = ImageDraw.Draw(img_draw)               # Crea un objeto para editar / dibujar sobre la imagen

radius = 2                                    # Radio de la elipse (puede ajustarse a conveniencia)

for point in points:                          # Recorre todas las inferencias encontradas por el modelo

    # Coordenadas del punto (inferencias)
    x = points["x"]
    y = points["y"]

                                              # Dibuja la elipse a partir de las coordenadas y radio dado
    draw.ellipse(
        [
            (x - radius, y - radius),
            (x + radius, x + radius)
        ],
        outline="yellow",                    # Color de la elipse (puede ajustarse a conveniencia)
        width=4                              # Grosor del trazo (puede ajustarse a conveniencia)
    )

plt.figure(figsize=(12,12))                  # Mostrar la imagen con los resultados
plt.imshow(img_draw)
plt.axis("off")
plt.show()  